In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

print("\nData load ho raha hai...")
df = pd.read_csv("/content/drive/MyDrive/hadoop_openstack/hadoop_logs_parsed.csv")
print(f"Shape: {df.shape}")

# Features
le_app = LabelEncoder()
le_container = LabelEncoder()
df['app_id_enc'] = le_app.fit_transform(df['app_id'])
df['container_id_enc'] = le_container.fit_transform(df['container_id'])

features = ['cluster_id', 'app_id_enc', 'container_id_enc']
X = df[features].values
y = df['label'].values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

X_train_normal = X_train[y_train == 0]

print(f"\nX_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Normal train: {len(X_train_normal):,}")
print("\nSab ready hai! ✅")

Mounted at /content/drive
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Data load ho raha hai...
Shape: (393433, 7)

X_train: (314746, 3)
X_test: (78687, 3)
Normal train: 20,228

Sab ready hai! ✅


In [2]:
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Layer, LeakyReLU, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import numpy as np

results_gen = {}

# ============================================
# Model 1: VAE
# ============================================
print("1. VAE training shuru...")

input_dim = 3
latent_dim = 2

class Sampling(Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Encoder
inputs_vae = Input(shape=(input_dim,))
h = Dense(16, activation='relu')(inputs_vae)
z_mean = Dense(latent_dim)(h)
z_log_var = Dense(latent_dim)(h)
z = Sampling()([z_mean, z_log_var])
encoder = Model(inputs_vae, [z_mean, z_log_var, z], name='encoder')

# Decoder
latent_inputs = Input(shape=(latent_dim,))
h_decoded = Dense(16, activation='relu')(latent_inputs)
outputs_vae = Dense(input_dim, activation='linear')(h_decoded)
decoder = Model(latent_inputs, outputs_vae, name='decoder')

# VAE
class VAE(Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, x):
        z_mean, z_log_var, z = self.encoder(x)
        reconstructed = self.decoder(z)
        kl_loss = -0.5 * tf.reduce_mean(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        self.add_loss(kl_loss)
        return reconstructed

vae = VAE(encoder, decoder)
vae.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True)

history_vae = vae.fit(
    X_train_normal, X_train_normal,
    epochs=20, batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop], verbose=1)

# Evaluate VAE
X_test_recon = vae(X_test, training=False).numpy()
mse_vae = np.mean(np.power(X_test - X_test_recon, 2), axis=1)
threshold_vae = np.percentile(mse_vae[y_test==0], 95)
y_pred_vae = (mse_vae > threshold_vae).astype(int)
roc_vae = roc_auc_score(y_test, mse_vae)

results_gen['VAE'] = {
    'roc_auc': roc_vae,
    'report': classification_report(y_test, y_pred_vae,
                target_names=['Normal', 'Anomaly'])
}
print(f"VAE ROC-AUC: {roc_vae:.4f} ✅")

# ============================================
# Model 2: GAN
# ============================================
print("\n2. GAN training shuru...")

latent_dim_gan = 16

def build_generator():
    model = Sequential([
        Dense(16, input_dim=latent_dim_gan),
        LeakyReLU(0.2),
        Dense(32),
        LeakyReLU(0.2),
        Dense(input_dim, activation='linear')
    ], name='generator')
    return model

def build_discriminator():
    model = Sequential([
        Dense(32, input_dim=input_dim),
        LeakyReLU(0.2),
        Dropout(0.3),
        Dense(16),
        LeakyReLU(0.2),
        Dense(1, activation='sigmoid')
    ], name='discriminator')
    return model

generator = build_generator()
discriminator = build_discriminator()
discriminator.compile(optimizer=Adam(0.0002),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])

discriminator.trainable = False
gan_input = Input(shape=(latent_dim_gan,))
gan_output = discriminator(generator(gan_input))
gan = Model(gan_input, gan_output, name='gan')
gan.compile(optimizer=Adam(0.0002), loss='binary_crossentropy')

EPOCHS = 50
BATCH_SIZE = 512

for epoch in range(EPOCHS):
    idx = np.random.randint(0, X_train_normal.shape[0], BATCH_SIZE)
    real_samples = X_train_normal[idx]
    noise = np.random.normal(0, 1, (BATCH_SIZE, latent_dim_gan))
    fake_samples = generator.predict(noise, verbose=0)

    real_labels = np.ones((BATCH_SIZE, 1))
    fake_labels = np.zeros((BATCH_SIZE, 1))

    d_loss_real = discriminator.train_on_batch(real_samples, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)

    noise2 = np.random.normal(0, 1, (BATCH_SIZE, latent_dim_gan))
    g_loss = gan.train_on_batch(noise2, real_labels)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"D_real: {d_loss_real[0]:.4f} | "
              f"D_fake: {d_loss_fake[0]:.4f} | "
              f"G_loss: {g_loss:.4f}")

# Evaluate GAN
d_scores = discriminator.predict(X_test, batch_size=512, verbose=0).flatten()
anomaly_scores = 1 - d_scores
threshold_gan = np.percentile(anomaly_scores[y_test==0], 95)
y_pred_gan = (anomaly_scores > threshold_gan).astype(int)
roc_gan = roc_auc_score(y_test, anomaly_scores)

results_gen['GAN'] = {
    'roc_auc': roc_gan,
    'report': classification_report(y_test, y_pred_gan,
                target_names=['Normal', 'Anomaly'])
}
print(f"\nGAN ROC-AUC: {roc_gan:.4f} ✅")

# ============================================
# Final Results
# ============================================
print("\n" + "=" * 45)
print("HADOOP — GENERATIVE MODELS RESULTS")
print("=" * 45)
for model, res in results_gen.items():
    print(f"\n{model} — ROC-AUC: {res['roc_auc']:.4f}")
    print(res['report'])

1. VAE training shuru...
Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 85ms/step - loss: 2.1149 - val_loss: 1.7955
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.7879 - val_loss: 1.5782
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6251 - val_loss: 1.4614
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.5111 - val_loss: 1.3412
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.4255 - val_loss: 1.2953
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.3836 - val_loss: 1.2486
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.3489 - val_loss: 1.2256
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.3263 - val_loss: 1.1942
Epoch 9/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.2888 - val_loss: 1.1658
Epoch 10/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.2355 - val_loss: 1.1277
Epoch 11/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.1633 - val_loss: 1.0613
Epoch 12/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/s

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Epoch 10/50 | D_real: 0.6149 | D_fake: 0.6202 | G_loss: 0.6753
Epoch 20/50 | D_real: 0.6193 | D_fake: 0.6223 | G_loss: 0.6702
Epoch 30/50 | D_real: 0.6221 | D_fake: 0.6243 | G_loss: 0.6657
Epoch 40/50 | D_real: 0.6261 | D_fake: 0.6277 | G_loss: 0.6611
Epoch 50/50 | D_real: 0.6298 | D_fake: 0.6312 | G_loss: 0.6560

GAN ROC-AUC: 0.6133 ✅

HADOOP — GENERATIVE MODELS RESULTS

VAE — ROC-AUC: 0.5235
              precision    recall  f1-score   support

      Normal       0.07      0.95      0.12      5057
     Anomaly       0.95      0.07      0.12     73630

    accuracy                           0.12     78687
   macro avg       0.51      0.51      0.12     78687
weighted avg       0.89      0.12      0.12     78687


GAN — ROC-AUC: 0.6133
              precision    recall  f1-score   support

      Normal       0.06      0.95      0.12      5057
     Anomaly       0.81      0.01      0.03     73630

    accuracy                           0.07     78687
   macro avg       0.44      0.48  

In [3]:
import json, os

save_dir = "/content/drive/MyDrive/hadoop_openstack"

hadoop_gen_results = {
    "VAE": {"ROC_AUC": 0.5235, "Recall": 0.07, "F1": 0.12},
    "GAN": {"ROC_AUC": 0.6133, "Recall": 0.01, "F1": 0.03}
}

with open(f"{save_dir}/hadoop_generative_results.json", "w") as f:
    json.dump(hadoop_gen_results, f, indent=4)

print("Saved! ✅")

# Full Hadoop Summary
print("\n" + "=" * 55)
print("HADOOP — ALL MODELS COMPLETE SUMMARY")
print("=" * 55)
print(f"\n{'Model':<22} {'ROC-AUC':>10} {'Recall':>10} {'F1':>10}")
print("-" * 55)

all_hadoop = [
    ("Random Forest",    1.0000, 1.00, 1.00),
    ("CNN Classifier",   0.9617, 0.84, 0.91),
    ("LSTM Autoencoder", 0.6519, 0.29, 0.45),
    ("GAN",              0.6133, 0.01, 0.03),
    ("VAE",              0.5235, 0.07, 0.12),
    ("One-Class SVM",    0.2383, 0.20, 0.32),
    ("Isolation Forest", 0.2092, 0.46, 0.60),
]

for model, roc, recall, f1 in all_hadoop:
    print(f"{model:<22} {roc:>10.4f} {recall:>10.2f} {f1:>10.2f}")

Saved! ✅

HADOOP — ALL MODELS COMPLETE SUMMARY

Model                     ROC-AUC     Recall         F1
-------------------------------------------------------
Random Forest              1.0000       1.00       1.00
CNN Classifier             0.9617       0.84       0.91
LSTM Autoencoder           0.6519       0.29       0.45
GAN                        0.6133       0.01       0.03
VAE                        0.5235       0.07       0.12
One-Class SVM              0.2383       0.20       0.32
Isolation Forest           0.2092       0.46       0.60
